[<img src="https://gitlab.irit.fr/toc/etu-n7/controle-optimal/-/raw/master/ressources/Logo-toulouse-inp-N7.png" alt="N7" height="80"/>](https://gitlab.irit.fr/toc/etu-n7/controle-optimal)
<img src="https://gitlab.irit.fr/toc/ens-n7/texCoursN7/-/raw/main/logo-insa.png" alt="INSA" height="80" style="margin-left:50px"/>

# Estimation de paramètres dans les EDO

- Date : 2026-2027
- Durée approximative : 2h

**Objectifs.** À l'issue de ce TP, vous savez :

- écrire un problème d'estimation de paramètres comme la minimisation d'un coût
  $J(p)$, et le résoudre par descente de gradient ;
- calculer $\partial x/\partial p$ **sans formule explicite de la solution**, en
  intégrant le système augmenté des équations variationnelles ;
- calculer le même gradient par la **méthode adjointe**, en résolvant une équation
  adjointe à rebours du temps ;
- dire laquelle des deux méthodes choisir selon le nombre de paramètres, et
  reconnaître dans l'équation adjointe l'ancêtre continu de la
  **rétropropagation**.

**Plan.**

1. Un modèle donné, un coût, une descente de gradient
2. Et si on n'avait pas la formule ?
3. L'équation variationnelle et le système augmenté
4. Lotka-Volterra : quand il n'y a plus de formule du tout
5. La méthode adjointe
6. Remarque : Euler explicite, réseaux de neurones et contrôle optimal
7. Le prix des deux méthodes
8. Annexe — pourquoi « adjointe », et d'où vient le gain ?

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Introduction
</div>

Jusqu'ici, les paramètres d'une EDO étaient **donnés** : on fixait $g$, $l$, $k$ pour
le pendule, $\alpha,\beta,\gamma,\delta$ pour Lotka-Volterra, et on regardait ce que
le modèle produisait. Dans la pratique c'est souvent l'inverse : on **observe** un
système, on connaît la forme de ses équations, et on cherche les paramètres qui
reproduisent le mieux les observations. C'est le **problème inverse**, ou problème
d'estimation de paramètres.

**Le cadre, commun à tout le TP.** On dispose de $K$ conditions initiales connues
$x_0^{(1)},\dots,x_0^{(K)}$. Pour chacune, on a observé l'état du système au **même**
instant final $T$ : une donnée $y^{(k)}$. Un **seul** jeu de paramètres $p$ est
partagé par toutes les trajectoires, et on cherche celui qui minimise

$$
J(p) \;=\; \sum_{k=1}^{K} \frac{1}{2}\,\big\|\,x\big(T;x_0^{(k)},p\big) - y^{(k)}\,\big\|^2 .
$$

Cette forme — plusieurs entrées, une même dynamique paramétrée, une somme d'erreurs
à la sortie — est exactement celle de l'apprentissage d'un réseau de neurones : les
$x_0^{(k)}$ jouent le rôle des exemples d'un lot, $p$ celui des poids partagés, et
$J$ celui de la fonction de perte. On y reviendra en section 6.

Les données $y^{(k)}$ seront **synthétiques** : on les fabrique en simulant le modèle
avec un paramètre « vrai » $p^\star$, puis en ajoutant un peu de bruit. On connaît
donc la réponse, ce qui permet de vérifier que la méthode la retrouve.

**Une règle pour tout le TP.** Les dérivées $\partial_x f$ et $\partial_p f$ du second
membre se calculent **à la main**, puis se codent. Pas de différences finies, pas de
différentiation automatique : ce sont les objets qu'on cherche justement à
comprendre.

In [ ]:
# activation du projet situé dans le répertoire de ce notebook
using Pkg
Pkg.activate(@__DIR__)

# chargement des paquets
using BenchmarkTools
using LinearAlgebra
using OrdinaryDiffEqTsit5
using Plots
using Printf
using Random

# figures vectorielles (nettes à tout zoom)
default(fmt = :svg)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
1. Un modèle donné, un coût, une descente de gradient
</div>

On se donne un modèle à un paramètre $p \in \mathbb{R}$ :

$$
x(t;x_0,p) \;=\; x_0\,e^{pt}.
$$

Pour l'instant, d'où il sort n'a aucune importance — c'est une formule, on sait
l'évaluer et on sait la dériver. On observe $K$ trajectoires partant de conditions
initiales $x_0^{(k)}$ connues, chacune mesurée au même instant $T$, et on cherche le
paramètre $p$ qui explique le mieux ces mesures :

$$
J(p) \;=\; \sum_{k=1}^{K} \frac{1}{2}\,\big(x_0^{(k)}e^{pT} - y^{(k)}\big)^2 .
$$

1. Calculer $J'(p)$ **à la main**, puis coder $J$ et $J'$ dans la cellule prévue.
2. Exécuter la descente de gradient (elle est fournie : c'est une méthode que vous
   connaissez déjà, il n'y a rien à y écrire) et vérifier qu'on retrouve $p^\star$.
3. Sur les deux graphiques, observer l'ajustement du modèle aux données (à droite) et
   repérer les itérés sur la courbe de $J$ (à gauche). La descente ralentit-elle en
   approchant du minimum ? Pourquoi ?


In [ ]:
"""
    modele(t, x0, p)

Modèle donné : la valeur de l'état à l'instant `t`, partant de `x0`, pour le
paramètre `p`.
"""
modele(t, x0, p) = x0 * exp(p * t)

In [ ]:
# --- données synthétiques (fournies) ---
# on simule le modèle avec le « vrai » paramètre, puis on ajoute du bruit de mesure
Random.seed!(1234)

p_vrai  = -0.8      # le paramètre que la descente devra retrouver
T       =  2.0      # instant d'observation, commun à toutes les trajectoires
K       =  20       # nombre de conditions initiales
σ_bruit =  0.02     # écart-type du bruit de mesure

x0s = collect(range(0.5, 3.0, length = K))                      # conditions initiales
ys  = [modele(T, x0, p_vrai) + σ_bruit * randn() for x0 ∈ x0s]   # observations en t = T

@printf("K = %d observations à t = %.1f, bruit σ = %.3f\n", K, T, σ_bruit)

In [ ]:
"""
    cout(p, x0s, ys, T)

Coût J(p) = Σₖ ½ (x(T; x0ᵏ, p) − yᵏ)².
"""
function cout(p, x0s, ys, T)
    return 0.0                                  # à compléter : le coût J(p)
end

"""
    grad_cout(p, x0s, ys, T)

Dérivée J'(p), obtenue en dérivant la formule du modèle à la main.
"""
function grad_cout(p, x0s, ys, T)
    return 0.0                                  # à compléter : J'(p), dérivé à la main
end;

In [ ]:
"""
    descente_gradient(∇J, p0; pas, itermax, tol)

Descente de gradient à pas fixe : p ← p − pas · ∇J(p). Retourne le dernier itéré et
l'historique complet. Fonctionne aussi bien pour un paramètre scalaire que vectoriel.

Cette fonction est **fournie** : la descente de gradient n'est pas l'objet du TP.
"""
function descente_gradient(∇J, p0; pas = 1e-3, itermax = 2000, tol = 1e-10)
    p = copy(p0)
    historique = [copy(p)]
    for _ ∈ 1:itermax
        g = ∇J(p)
        norm(g) < tol && break
        p = p - pas * g
        push!(historique, copy(p))
    end
    return p, historique
end;

In [ ]:
# descente à partir d'un paramètre initial volontairement faux
p_initial = -0.2
p_estime, historique = descente_gradient(p -> grad_cout(p, x0s, ys, T), p_initial;
                                         pas = 2e-3, itermax = 2000)

@printf("p initial : %8.4f\n", p_initial)
@printf("p estimé  : %8.4f\n", p_estime)
@printf("p vrai    : %8.4f\n", p_vrai)
@printf("itérations : %d\n", length(historique) - 1)

In [ ]:
# à gauche : le coût le long de l'axe des paramètres et les itérés de la descente
# à droite : les observations en t = T face au modèle initial et au modèle estimé
pgrid = range(-1.5, 0.0, length = 300)
plt_cout = plot(pgrid, [cout(p, x0s, ys, T) for p ∈ pgrid],
                xlabel = "p", ylabel = "J(p)", label = "J", lw = 2)
scatter!(plt_cout, historique, [cout(p, x0s, ys, T) for p ∈ historique],
         ms = 2, label = "itérés", color = :red)
vline!(plt_cout, [p_vrai], ls = :dash, color = :black, label = "p*")

plt_obs = scatter(x0s, ys, xlabel = "x₀", ylabel = "état en t = T",
                  label = "observations yᵏ", color = :black, ms = 3.5)
plot!(plt_obs, x0s, [modele(T, x0, p_initial) for x0 ∈ x0s],
      ls = :dash, lw = 2, color = :gray, label = "modèle initial")
plot!(plt_obs, x0s, [modele(T, x0, p_estime) for x0 ∈ x0s],
      lw = 2, color = :red, label = "modèle estimé")

plot(plt_cout, plt_obs, layout = (1, 2), size = (950, 350))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
2. Et si on n'avait pas la formule ?
</div>

Tout ce qui précède repose sur un point : on connaissait $x(t;x_0,p) = x_0e^{pt}$, donc
on savait la dériver par rapport à $p$. Or cette formule n'est pas tombée du ciel —
c'est la solution du problème de Cauchy

$$
\dot x(t) = p\,x(t), \qquad x(0) = x_0 .
$$

Autrement dit, ce qu'on vient de faire, c'est de l'estimation de paramètres dans une
EDO **qui se résout à la main**. Mais les EDO qui se résolvent à la main sont
l'exception : dès que le second membre est un peu non linéaire, il n'y a plus de
formule, donc plus de dérivée à dériver.

> **La question du TP.** Comment calculer $\dfrac{\partial x}{\partial p}(T)$ quand on ne
> sait intégrer l'EDO que numériquement ?

Deux réponses, dans cet ordre : l'**équation variationnelle** (section 3), puis la
**méthode adjointe** (section 5). On les construit toutes les deux sur cet exemple
minuscule — où la réponse exacte est connue et permet donc de les valider — avant de
les lâcher sur un problème sans formule.

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
3. L'équation variationnelle et le système augmenté
</div>

Soit $\dot x(t) = f(x(t),p)$ avec $x(0)=x_0$ indépendant de $p$. On note

$$
S(t) \;\coloneqq\; \frac{\partial x}{\partial p}(t) \;\in\; \mathbb{R}^{n\times m}.
$$

En dérivant l'EDO par rapport à $p$ et en échangeant les dérivations (licite car $f$
est $\mathcal{C}^1$), il vient l'**équation variationnelle**

$$
\dot S(t) \;=\; \underbrace{\frac{\partial f}{\partial x}\big(x(t),p\big)}_{n\times n}\,S(t)
           \;+\; \underbrace{\frac{\partial f}{\partial p}\big(x(t),p\big)}_{n\times m},
\qquad S(0) = 0 ,
$$

la condition initiale étant nulle puisque $x_0$ ne dépend pas de $p$. Cette équation
fait intervenir $x(t)$ : on l'intègre donc **en même temps** que l'état, dans un seul
**système augmenté** d'inconnue $(x,S)$ :

$$
\left\{
\begin{aligned}
\dot x &= f(x,p), & x(0) &= x_0,\\
\dot S &= \partial_x f(x,p)\,S + \partial_p f(x,p), & S(0) &= 0 .
\end{aligned}
\right.
$$

Une seule intégration fournit donc à la fois $x(T)$ et $\partial_p x(T)$.

> Le polycopié traite cette dépendance par rapport à un paramètre au chapitre
> « Dépendance par rapport à un paramètre », et l'illustre sur un système $3\times3$
> qui, lui, se résout explicitement.

**Sur notre exemple.** Ici $n=m=1$ et $f(x,p) = p\,x$.

1. Calculer $\partial_x f$ et $\partial_p f$, puis coder le second membre augmenté.
2. Intégrer et comparer $S(T)$ à la dérivée exacte $\partial_p\big(x_0e^{pT}\big) = x_0\,T\,e^{pT}$.
3. Recalculer $\nabla J$ à partir de $S(T)$ — sans jamais utiliser la formule du modèle —
   et vérifier que la descente de gradient retrouve le même $p^\star$ qu'en section 1.


In [ ]:
"""
    expo_augmente(Z, p, t)

Second membre du système augmenté Z = (x, S) pour ẋ = p x :
    ẋ = p x,
    Ṡ = ∂ₓf · S + ∂ₚf.
"""
function expo_augmente(Z, p, t)
    x, S = Z
    return [0.0, 0.0]                           # à compléter : (ẋ, Ṡ) avec ∂ₓf = p et ∂ₚf = x
end;

In [ ]:
# --- validation : le système augmenté contre la dérivée exacte ---
p_essai  = -0.8
x0_essai =  1.5

prob = ODEProblem(expo_augmente, [x0_essai, 0.0], (0.0, T), p_essai,
                  reltol = 1e-10, abstol = 1e-10)
sol  = solve(prob, Tsit5())
xT, ST = sol(T)

S_exact = x0_essai * T * exp(p_essai * T)   # dérivée de la formule, calculée à la main

@printf("S(T) par le système augmenté : %.12f\n", ST)
@printf("S(T) exact                   : %.12f\n", S_exact)
@printf("écart                        : %.2e\n", abs(ST - S_exact))

In [ ]:
"""
    grad_sensibilites_expo(p, x0s, ys, T)

Gradient de J obtenu par le système augmenté : une intégration par condition
initiale, qui fournit d'un coup x(T) et S(T) = ∂ₚx(T). La formule du modèle n'est
jamais utilisée.
"""
function grad_sensibilites_expo(p, x0s, ys, T)
    g = 0.0
    for (x0, y) ∈ zip(x0s, ys)
        prob = ODEProblem(expo_augmente, [x0, 0.0], (0.0, T), p,
                          reltol = 1e-10, abstol = 1e-10)
        sol  = solve(prob, Tsit5())
        xT, ST = sol(T)
        g += 0.0                                # à compléter : contribution de cet échantillon
    end
    return g
end;

In [ ]:
# les deux gradients coïncident-ils ?
p_essai = -0.5
@printf("gradient à la main      : %.10f\n", grad_cout(p_essai, x0s, ys, T))
@printf("gradient par sensibilité: %.10f\n", grad_sensibilites_expo(p_essai, x0s, ys, T))

# et la descente donne-t-elle le même résultat ?
p_estime_sens, _ = descente_gradient(p -> grad_sensibilites_expo(p, x0s, ys, T), p_initial;
                                     pas = 2e-3, itermax = 2000)
@printf("\np estimé (section 1)    : %8.4f\n", p_estime)
@printf("p estimé (sensibilités) : %8.4f\n", p_estime_sens)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
4. Lotka-Volterra : quand il n'y a plus de formule du tout
</div>

On reprend le modèle proie-prédateur du TP précédent,

$$
\left\{
\begin{aligned}
\dot x_1 &= \alpha x_1 - \beta x_1 x_2,\\
\dot x_2 &= \delta x_1 x_2 - \gamma x_2,
\end{aligned}
\right.
\qquad q = (\alpha,\beta,\gamma,\delta),
$$

dont aucune solution explicite n'est connue : la section 1 est donc hors-jeu, et le
système augmenté devient le seul moyen d'obtenir $\partial x/\partial p$.

On suppose $\beta$ et $\delta$ connus, et on estime $p = (\alpha,\gamma)$ : deux
paramètres, ce qui permettra de **dessiner** le paysage du coût. L'horizon $T$ est
volontairement court (une fraction de période) pour que le coût reste bien conditionné.

> Attention aux notations : dans ce notebook $q$ désigne le vecteur des paramètres du
> modèle et $p$ le sous-vecteur à estimer, car la lettre $\lambda$ est réservée au
> **vecteur adjoint** de la section 5.

1. Calculer à la main $A(x) = \partial_x f$ (matrice $2\times2$) et
   $B(x) = \partial_p f$ (matrice $2\times2$, pour $p = (\alpha,\gamma)$), puis les coder.
2. Coder le second membre du système augmenté $(x,S)$ avec $S\in\mathbb{R}^{2\times2}$.
3. Assembler $\nabla J = \sum_k S^{(k)}(T)^\top\big(x^{(k)}(T)-y^{(k)}\big)$ et lancer la descente.
4. Sur la figure : vérifier à droite que les états finaux prédits rejoignent les
   observations, et observer à gauche les lignes de niveau de $J$. Sont-elles plutôt
   circulaires ou allongées ? Qu'en déduire sur le conditionnement du problème, et
   comment cela se lit-il sur la trajectoire de la descente ?


In [ ]:
"""
    lv(x, q, t)

Second membre de Lotka-Volterra, q = (α, β, γ, δ).
"""
function lv(x, q, t)
    α, β, γ, δ = q
    return [α*x[1] - β*x[1]*x[2],
            δ*x[1]*x[2] - γ*x[2]]
end

# β et δ sont supposés connus ; seuls p = (α, γ) sont à estimer
β_connu = 4/3
δ_connu = 1.0

"""
    q_de(p)

Reconstruit le vecteur complet des paramètres q = (α, β, γ, δ) à partir de p = (α, γ).
"""
q_de(p) = [p[1], β_connu, p[2], δ_connu];

In [ ]:
"""
    jacobienne_A(x, q)

A(x) = ∂f/∂x, matrice 2×2.
"""
function jacobienne_A(x, q)
    α, β, γ, δ = q
    return zeros(2, 2)                          # à compléter : ∂f/∂x
end

"""
    jacobienne_B(x, q)

B(x) = ∂f/∂p pour p = (α, γ), matrice 2×2.
"""
function jacobienne_B(x, q)
    return zeros(2, 2)                          # à compléter : ∂f/∂p pour p = (α, γ)
end;

In [ ]:
"""
    lv_augmente(Z, p, t)

Second membre du système augmenté Z = (x, vec(S)) pour Lotka-Volterra :
    ẋ = f(x, q),    Ṡ = A(x) S + B(x),    S ∈ ℝ^{2×2}, S(0) = 0.
"""
function lv_augmente(Z, p, t)
    q = q_de(p)
    x = Z[1:2]
    S = reshape(Z[3:6], 2, 2)
    dx = zeros(2); dS = zeros(2, 2)             # à compléter : ẋ et Ṡ
    return vcat(dx, vec(dS))
end;

In [ ]:
# --- données synthétiques (fournies) ---
Random.seed!(4321)

p_vrai_lv = [2/3, 1.0]      # (α, γ) à retrouver
T_lv      = 1.0             # horizon court
K_lv      = 12              # nombre de conditions initiales

x0s_lv = [[1.0 + 0.15i, 0.8 + 0.10i] for i ∈ 0:K_lv-1]

function simule_lv(x0, p, T)
    prob = ODEProblem(lv, x0, (0.0, T), q_de(p), reltol = 1e-10, abstol = 1e-10)
    return solve(prob, Tsit5())(T)
end

ys_lv = [simule_lv(x0, p_vrai_lv, T_lv) + 0.01 * randn(2) for x0 ∈ x0s_lv]

"""
    cout_lv(p, x0s, ys, T; tol)

Coût J(p) pour Lotka-Volterra (même forme qu'en section 1, mais l'état s'obtient
maintenant par intégration numérique).
"""
function cout_lv(p, x0s, ys, T; tol = 1e-10)
    J = 0.0
    for (x0, y) ∈ zip(x0s, ys)
        prob = ODEProblem(lv, x0, (0.0, T), q_de(p), reltol = tol, abstol = tol)
        xT   = solve(prob, Tsit5())(T)
        J   += 0.5 * sum(abs2, xT - y)
    end
    return J
end;

In [ ]:
"""
    grad_sensibilites_lv(p, x0s, ys, T)

Gradient de J par le système augmenté : une intégration de (x, S) par condition
initiale, puis ∇J = Σₖ S⁽ᵏ⁾(T)ᵀ (x⁽ᵏ⁾(T) − y⁽ᵏ⁾).
"""
function grad_sensibilites_lv(p, x0s, ys, T)
    g = zeros(2)
    for (x0, y) ∈ zip(x0s, ys)
        Z0   = vcat(x0, zeros(4))            # S(0) = 0
        prob = ODEProblem(lv_augmente, Z0, (0.0, T), p, reltol = 1e-10, abstol = 1e-10)
        ZT   = solve(prob, Tsit5())(T)
        xT   = ZT[1:2]
        ST   = reshape(ZT[3:6], 2, 2)
        g += zeros(2)                           # à compléter : contribution de cet échantillon
    end
    return g
end;

In [ ]:
# le pas et le nombre d'itérations peuvent être ajustés si la convergence est trop lente
p_initial_lv = [0.45, 1.25]
p_estime_lv, hist_lv = descente_gradient(p -> grad_sensibilites_lv(p, x0s_lv, ys_lv, T_lv),
                                         p_initial_lv; pas = 2e-3, itermax = 1000)

@printf("p initial : α = %.4f, γ = %.4f   (J = %.3e)\n",
        p_initial_lv[1], p_initial_lv[2], cout_lv(p_initial_lv, x0s_lv, ys_lv, T_lv))
@printf("p estimé  : α = %.4f, γ = %.4f   (J = %.3e)\n",
        p_estime_lv[1], p_estime_lv[2], cout_lv(p_estime_lv, x0s_lv, ys_lv, T_lv))
@printf("p vrai    : α = %.4f, γ = %.4f   (J = %.3e)\n",
        p_vrai_lv[1], p_vrai_lv[2], cout_lv(p_vrai_lv, x0s_lv, ys_lv, T_lv))
@printf("itérations : %d\n", length(hist_lv) - 1)

In [ ]:
# à gauche : le paysage du coût dans le plan des paramètres et le chemin suivi par la descente
# à droite : les observations en t = T face aux états finaux prédits (modèle initial vs estimé)
αs = range(0.40, 0.95, length = 30)
γs = range(0.70, 1.30, length = 30)
Jgrid = [cout_lv([a, c], x0s_lv, ys_lv, T_lv; tol = 1e-6) for c ∈ γs, a ∈ αs]

plt_contour = contour(αs, γs, Jgrid, levels = 25, xlabel = "α", ylabel = "γ", colorbar_title = "J")
plot!(plt_contour, first.(hist_lv), last.(hist_lv), lw = 2, color = :red, label = "descente")
scatter!(plt_contour, [p_initial_lv[1]], [p_initial_lv[2]], color = :red,   label = "départ")
scatter!(plt_contour, [p_vrai_lv[1]],    [p_vrai_lv[2]],    color = :black, label = "p*")

y_init_lv = [simule_lv(x0, p_initial_lv, T_lv) for x0 ∈ x0s_lv]
y_est_lv  = [simule_lv(x0, p_estime_lv,  T_lv) for x0 ∈ x0s_lv]
plt_obs_lv = scatter(first.(ys_lv), last.(ys_lv),
                     xlabel = "x₁(T) (proies)", ylabel = "x₂(T) (prédateurs)",
                     color = :black, ms = 4, label = "observations yᵏ")
plot!(plt_obs_lv, first.(y_init_lv), last.(y_init_lv), ls = :dash, marker = :circle, ms = 3,
      color = :gray, label = "modèle initial")
plot!(plt_obs_lv, first.(y_est_lv), last.(y_est_lv), lw = 2, marker = :circle, ms = 3,
      color = :red, label = "modèle estimé")

plot(plt_contour, plt_obs_lv, layout = (1, 2), size = (950, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
5. La méthode adjointe
</div>

Le système augmenté a un défaut qu'on ne voit pas encore ici : $S$ est une matrice
$n\times m$, donc il ajoute $n\,m$ équations. Avec $m=2$ c'est indolore ; avec $m$
grand — un réseau de neurones en compte des millions — c'est rédhibitoire. Or on ne
veut pas $S(T)$ tout entière : on veut seulement le vecteur $\nabla J \in \mathbb{R}^m$.
La méthode adjointe l'obtient sans jamais former $S$.

**Construction.** On traite l'EDO comme une **contrainte** et on lui associe un
multiplicateur de Lagrange $\lambda(t)$ :

$$
\mathcal{L}(p) \;=\; J\big(x(T)\big) \;+\; \int_0^T \lambda(t)^\top\Big(f\big(x(t),p\big)-\dot x(t)\Big)\,\mathrm{d}t .
$$

La parenthèse est nulle le long de toute trajectoire admissible : $\mathcal{L} = J$
**quel que soit** $\lambda$. On est donc libre de choisir $\lambda$ comme cela nous
arrange. En faisant varier $p$ de $\delta p$, et en intégrant par parties le terme
$\int\lambda^\top\dot{x}$ (c'est là que tout se joue) :

$$
\delta\mathcal{L}
= \underbrace{\Big(\nabla_x J\big(x(T)\big)-\lambda(T)\Big)^\top \delta x(T)}_{\text{terme de bord}}
+ \int_0^T \underbrace{\Big(\partial_x f^\top\lambda+\dot\lambda\Big)^\top}_{\text{terme en } \delta x(t)} \delta x(t)\,\mathrm{d}t
+ \int_0^T \lambda^\top\,\partial_p f\;\delta p\,\mathrm{d}t .
$$

Les deux premiers termes font intervenir $\delta x$, c'est-à-dire précisément la
sensibilité qu'on cherche à éviter de calculer. On **choisit donc $\lambda$ pour les
annuler** : c'est l'**équation adjointe**, avec sa condition à l'instant **final**,

$$
\boxed{\;-\dot\lambda(t) = \partial_x f\big(x(t),p\big)^\top \lambda(t),
\qquad \lambda(T) = x(T)-y\;}
$$

et il ne reste alors que

$$
\boxed{\;\nabla J \;=\; \int_0^T \partial_p f\big(x(t),p\big)^\top \lambda(t)\,\mathrm{d}t\;}
$$

**Ce que cela donne comme algorithme.** Trois temps, pour chaque condition initiale :

1. **passe directe** : intégrer $x$ de $0$ à $T$ et garder la trajectoire ;
2. **passe rétrograde** : intégrer $\lambda$ de $T$ à $0$, en partant de $\lambda(T)=x(T)-y$ ;
3. **accumulation** : intégrer $\partial_pf^\top\lambda$ le long du chemin.

Les étapes 2 et 3 se font en une seule intégration à rebours, d'inconnue $(\lambda,g)$
avec $g(T)=0$ et $\dot g = -\partial_p f^\top\lambda$, si bien que $g(0) = \nabla J$
pour l'échantillon considéré. Noter la taille de cet état : $n+m$, une **somme**, là
où le système augmenté avait $n + nm$, un **produit**.

La passe rétrograde a besoin de $x(t)$ : on réutilise l'interpolation dense `sol(t)`
de la passe directe, plutôt que de ré-intégrer l'état à l'envers.

1. Coder le second membre du système adjoint augmenté $(\lambda, g)$.
2. Assembler le gradient, et le **comparer à celui de la section 4** — les deux doivent
   coïncider à la précision du solveur.
3. Relancer la descente avec ce gradient : mêmes paramètres estimés.


In [ ]:
"""
    lv_adjoint(W, params, t)

Second membre du système adjoint augmenté W = (λ, g), intégré à rebours de T à 0 :
    λ̇ = −A(x(t))ᵀ λ,     ġ = −B(x(t))ᵀ λ.
Partant de g(T) = 0, on obtient ainsi g(0) = ∫₀ᵀ B(x)ᵀ λ dt, c'est-à-dire la
contribution de cet échantillon au gradient.

`params = (p, sol)` où `sol` est la solution de la passe directe : on y lit x(t).
"""
function lv_adjoint(W, params, t)
    p, sol = params
    q = q_de(p)
    x = sol(t)
    λ = W[1:2]
    dλ = zeros(2); dg = zeros(2)                # à compléter : λ̇ et ġ
    return vcat(dλ, dg)
end;

In [ ]:
"""
    grad_adjoint_lv(p, x0s, ys, T)

Gradient de J par la méthode adjointe : passe directe, puis passe rétrograde, pour
chaque condition initiale.
"""
function grad_adjoint_lv(p, x0s, ys, T)
    g = zeros(2)
    q = q_de(p)
    for (x0, y) ∈ zip(x0s, ys)
        # passe directe : on garde la solution pour pouvoir relire x(t)
        prob_f = ODEProblem(lv, x0, (0.0, T), q, reltol = 1e-10, abstol = 1e-10)
        sol_f  = solve(prob_f, Tsit5())
        xT     = sol_f(T)

        # passe rétrograde : de T vers 0, en partant de (λ(T), g(T))
        W_T = zeros(4)                          # à compléter : (λ(T), g(T))
        prob_b = ODEProblem(lv_adjoint, W_T, (T, 0.0), (p, sol_f),
                            reltol = 1e-10, abstol = 1e-10)
        sol_b  = solve(prob_b, Tsit5())

        g += sol_b(0.0)[3:4]
    end
    return g
end;

In [ ]:
# --- validation : l'adjoint contre les sensibilités ---
p_test = [0.55, 1.15]
g_sens = grad_sensibilites_lv(p_test, x0s_lv, ys_lv, T_lv)
g_adj  = grad_adjoint_lv(p_test, x0s_lv, ys_lv, T_lv)

@printf("∇J par sensibilités : [% .10f, % .10f]\n", g_sens[1], g_sens[2])
@printf("∇J par adjoint      : [% .10f, % .10f]\n", g_adj[1],  g_adj[2])
@printf("écart relatif       : %.2e\n", norm(g_sens - g_adj) / norm(g_sens))

In [ ]:
# la descente pilotée par le gradient adjoint doit retrouver les mêmes paramètres
p_estime_adj, _ = descente_gradient(p -> grad_adjoint_lv(p, x0s_lv, ys_lv, T_lv),
                                    p_initial_lv; pas = 2e-3, itermax = 1000)

@printf("p estimé (sensibilités) : α = %.4f, γ = %.4f\n", p_estime_lv[1],  p_estime_lv[2])
@printf("p estimé (adjoint)      : α = %.4f, γ = %.4f\n", p_estime_adj[1], p_estime_adj[2])
@printf("p vrai                  : α = %.4f, γ = %.4f\n", p_vrai_lv[1],    p_vrai_lv[2])

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
6. Remarque : Euler explicite, réseaux de neurones et contrôle optimal
</div>

Dans un réseau de neurones à $N$ couches comme dans une EDO discrétisée, deux types de
variables coexistent : l'**état** (ou activation) $a_k \in \mathbb{R}^n$, qui traverse le
calcul **horizontalement** d'étape en étape à partir de l'entrée $a_0 = x_0$, et les
**paramètres** $\beta_k \in \mathbb{R}^d$, qui entrent **verticalement** dans chaque
étape :

$$
a_0 = x_0, \qquad a_{k+1} = f_k(a_k,\beta_k) \quad (k=0,\dots,N-1), \qquad J = \ell(a_N) = \tfrac12\|a_N-y\|^2.
$$

```
                   λ₁ = A₁ᵀ λ₂     λ₂ = A₂ᵀ λ₃     λ₃ = ∇ℓ(a₃)
                   ◄────────────── ◄────────────── ◄──────────
        a₀ ──► f₀ ──────► a₁ ──► f₁ ──────► a₂ ──► f₂ ──────► a₃ ──► J(a₃)
               ▲                 ▲                 ▲
               │ β₀              │ β₁              │ β₂
               ▼ B₀ᵀ λ₁          ▼ B₁ᵀ λ₂          ▼ B₂ᵀ λ₃
```

Notons $A_k = \partial_a f_k \in \mathbb{R}^{n\times n}$ et $B_k = \partial_\beta f_k \in \mathbb{R}^{n\times d}$
les jacobiennes locales. Les deux modes de dérivation se lisent directement sur ce graphe :

- **En mode direct (*forward*, de gauche à droite $\longrightarrow$)** : une variation des
  paramètres engendre $\delta a_{k+1} = A_k\,\delta a_k + B_k\,\delta\beta_k$. Propager la
  sensibilité par rapport à toutes les directions de paramètres oblige à transporter des
  **matrices** de taille $n\times m$ à travers les $A_k$.
- **En mode adjoint (*backward* / rétropropagation, de droite à gauche $\longleftarrow$)** :
  on part de la sortie scalaire $J$ avec le seul **vecteur** $\lambda_N = \nabla\ell(a_N) = a_N - y \in \mathbb{R}^n$,
  qu'on remonte horizontalement d'étape en étape, puis qu'on projette verticalement vers
  les paramètres :
  $$
  \lambda_k = A_k^\top \lambda_{k+1}, \qquad \text{contribution de l'étape } k :\; B_k^\top \lambda_{k+1}.
  $$

Selon la nature de $\beta_k$, on obtient alors deux problèmes classiques :

1. **Un paramètre $\beta_k$ propre à chaque étape :**
   - en apprentissage profond, c'est un **réseau à couches indépendantes** (MLP, ResNet), où
     $\beta_k$ sont les poids de la couche $k$ ;
   - en automatique, avec $f_k(a_k,\beta_k) = a_k + h\,f(a_k, u(t_k))$, c'est un problème de
     **contrôle optimal** discrétisé, où $\beta_k = u(t_k)$ est la commande à l'instant $t_k$.
   Chaque $\beta_k$ n'agissant qu'à l'étape $k$, son gradient se lit **localement** sur sa
   flèche verticale, sans rien accumuler : $\nabla_{\beta_k} J = B_k^\top \lambda_{k+1}$.
2. **Un même paramètre $\beta_0 = \dots = \beta_{N-1} = p$ partagé par toutes les étapes :**
   - en apprentissage profond, c'est un **réseau récurrent (RNN)** ou une **Neural ODE** ;
   - en EDO, avec Euler explicite $f_k(a_k,p) = a_k + h\,f(a_k,p)$, c'est l'**estimation de
     paramètres constants** de ce TP.
   Puisque le même $p$ tire sur toutes les flèches verticales à la fois, la règle de la
   chaîne **additionne** toutes leurs contributions :
   $$
   \lambda_k = \big(I + h\,\partial_x f(a_k,p)\big)^{\!\top}\lambda_{k+1},
   \qquad
   \nabla_p J = \sum_{k=0}^{N-1} h\,\partial_p f(a_k,p)^\top \lambda_{k+1}.
   $$
   Quand $h \to 0$, cette récurrence rétrograde devient $-\dot\lambda = \partial_x f^\top\lambda$
   et la somme devient l'intégrale $\int_0^T \partial_p f^\top\lambda\,\mathrm{d}t$ de la
   section 5.

| régime | discret ($a_{k+1} = f_k(a_k,\beta_k)$) | continu ($h\to 0$) | gradient |
|---|---|---|---|
| **paramètres par étape** ($\beta_k$) | MLP / ResNet | **contrôle optimal** $\dot x = f(x,u(t))$ | local : $B_k^\top\lambda_{k+1}$ |
| **paramètres partagés** ($\beta_k = p$) | RNN / BPTT | **estimation de paramètres** $\dot x = f(x,p)$ (Neural ODE) | accumulé : $\sum_k B_k^\top\lambda_{k+1} \to \int_0^T B^\top\lambda\,\mathrm{d}t$ |

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
7. Le prix des deux méthodes
</div>

Les deux méthodes donnent le même gradient : reste à savoir laquelle coûte le moins
cher. Comparons les tailles des systèmes à intégrer, pour $n$ états et $m$ paramètres :

| méthode | état intégré | taille | coût d'un second membre |
|---|---|---|---|
| sensibilités | $(x,S)$, $S\in\mathbb{R}^{n\times m}$ | $n + nm$ (**produit**) | $AS + B$ : $\Theta(n^2 m)$ |
| adjoint | $x$ puis $(\lambda,g)$ | $n$ puis $n+m$ (**somme**) | $A^\top\lambda$ et $B^\top\lambda$ : $\Theta(n^2 + nm)$ |

À chaque pas, le second membre de $\dot S = AS+B$ coûte le produit matrice-matrice
$AS$, soit $\Theta(n^2m)$ opérations, quand celui de l'adjoint ne coûte que les produits
matrice-vecteur $A^\top\lambda$ et $B^\top\lambda$, soit $\Theta(n^2 + nm)$. **Les deux
charges croissent linéairement en $m$** — puisque les $m$ paramètres agissent à chaque
pas et qu'il faut bien accumuler les $m$ composantes de $\nabla J$ — mais celle de
l'adjoint croît **$n$ fois moins vite**.

**Et le nombre de données $K$ ?** Il n'arbitre rien : chaque échantillon demande sa
propre intégration dans les deux cas, donc le coût est proportionnel à $K$ **des deux
côtés**.

**Le vérifier.** Il faut un problème où $m$ est un curseur et où la dimension d'état
$n$ n'est pas minuscule. On prend une chaîne amortie de $n = 20$ états couplés, forcée
sur sa dernière composante par une combinaison de $m$ fonctions de base lisses :

$$
\dot x = A\,x + B(t)\,p, \qquad (B(t)\,p)_n = \sum_{j=1}^{m} p_j\,\varphi_j(t),
\qquad \varphi_j(t) = \frac{\sin(jt)}{j^3}.
$$

Le coefficient $1/j^3$ amortit les hautes fréquences afin que le solveur adaptatif
fasse le **même nombre de pas** quel que soit $m$. Les jacobiennes $\partial_x f = A$
(constante, $n\times n$) et $\partial_p f = B(t)$ ($n\times m$) sont fournies — l'objet
de cette section est la **mesure**, pas le calcul. On fixe $K=1$ et on mesure avec
`BenchmarkTools`.

1. Compléter la boucle de mesure.
2. Lire le graphique : quelle est la pente de chaque courbe en échelle log-log ? Quel
   écart observe-t-on entre les deux courbes pour $m$ grand, et est-ce conforme au
   tableau ci-dessus ?
3. Pourquoi le gain entre mode adjoint (rétropropagation) et mode direct atteint-il un
   facteur $m$ (des millions) dans un réseau à couches indépendantes, alors qu'il
   plafonne à un facteur $\approx n$ sur ce système ?


In [ ]:
# --- le système couplé de taille n = 20, à m ajustable, et ses jacobiennes (fournis) ---

const n_osc = 20
const A_OSC = Matrix(Tridiagonal(ones(n_osc - 1), -2.1 * ones(n_osc), ones(n_osc - 1)))

φ(j, t) = sin(j * t) / j^3

"""
    oscillateur(x, p, t)

Chaîne amortie de `n_osc = 20` états couplés, forcée par `m = length(p)` fonctions de base.
"""
function oscillateur(x, p, t)
    dx = A_OSC * x
    dx[end] += sum(p[j] * φ(j, t) for j ∈ eachindex(p))
    return dx
end

"""
    B_osc(t, m)

∂f/∂p, matrice n_osc × m : le forçage agit sur la dernière composante.
"""
function B_osc(t, m)
    B = zeros(n_osc, m)
    for j ∈ 1:m
        B[end, j] = φ(j, t)
    end
    return B
end;

In [ ]:
# --- les deux gradients pour ce système (fournis : même structure qu'aux sections 4 et 5) ---

function osc_augmente(Z, p, t)
    m  = length(p)
    x  = Z[1:n_osc]
    S  = reshape(Z[n_osc+1:end], n_osc, m)
    dx = oscillateur(x, p, t)
    dS = A_OSC * S + B_osc(t, m)
    return vcat(dx, vec(dS))
end

function osc_adjoint(W, params, t)
    p, _ = params
    m  = length(p)
    λ  = W[1:n_osc]
    dλ = -A_OSC' * λ
    dg = -B_osc(t, m)' * λ
    return vcat(dλ, dg)
end

function grad_sensibilites_osc(p, x0, y, T)
    m    = length(p)
    prob = ODEProblem(osc_augmente, vcat(x0, zeros(n_osc * m)), (0.0, T), p,
                      reltol = 1e-8, abstol = 1e-8)
    ZT   = solve(prob, Tsit5())(T)
    return reshape(ZT[n_osc+1:end], n_osc, m)' * (ZT[1:n_osc] - y)
end

function grad_adjoint_osc(p, x0, y, T)
    m      = length(p)
    prob_f = ODEProblem(oscillateur, x0, (0.0, T), p, reltol = 1e-8, abstol = 1e-8)
    sol_f  = solve(prob_f, Tsit5())
    prob_b = ODEProblem(osc_adjoint, vcat(sol_f(T) - y, zeros(m)), (T, 0.0), (p, sol_f),
                        reltol = 1e-8, abstol = 1e-8)
    return solve(prob_b, Tsit5())(0.0)[n_osc+1:end]
end;

In [ ]:
# --- vérification préalable : les deux gradients coïncident-ils sur ce système aussi ? ---
T_osc  = 5.0
x0_osc = ones(n_osc)
y_osc  = 0.5 * ones(n_osc)
p_osc  = 0.1 * ones(8)

g_s = grad_sensibilites_osc(p_osc, x0_osc, y_osc, T_osc)
g_a = grad_adjoint_osc(p_osc, x0_osc, y_osc, T_osc)
@printf("m = %d, écart relatif entre les deux gradients : %.2e\n",
        length(p_osc), norm(g_s - g_a) / norm(g_s))

In [ ]:
# --- mesure des deux coûts en fonction de m ---
ms = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

temps_sens = Float64[]
temps_adj  = Float64[]

for m ∈ ms
    # `global` : BenchmarkTools interpole ($p_b) une variable globale sans ambiguïté
    global p_b = 0.1 * ones(m)
    push!(temps_sens, 0.0); push!(temps_adj, 0.0)   # à compléter : mesurer avec @belapsed
    @printf("m = %4d : sensibilités %.2e s, adjoint %.2e s  (×%.1f)\n",
            m, temps_sens[end], temps_adj[end], temps_sens[end] / temps_adj[end])
end

In [ ]:
plt = plot(ms, temps_sens, marker = :circle, lw = 2, label = "sensibilités (forward)",
           xlabel = "m (nombre de paramètres)", ylabel = "temps d'un gradient (s)",
           xscale = :log10, yscale = :log10, legend = :topleft, size = (650, 400))
plot!(plt, ms, temps_adj, marker = :square, lw = 2, label = "adjoint (reverse)")

**Conclusion.** Dès que $m$ dépasse quelques unités, la méthode adjointe l'emporte
nettement : remplacer la propagation de la matrice $S \in \mathbb{R}^{n\times m}$
(produit matrice-matrice $AS$ en $\Theta(n^2 m)$) par celle du seul vecteur
$\lambda \in \mathbb{R}^n$ (produits matrice-vecteur $A^\top\lambda$ et $B^\top\lambda$ en
$\Theta(n^2 + nm)$) divise le temps de calcul par un facteur d'ordre $n$.

Deux remarques importantes :

- Sur un système non linéaire comme Lotka-Volterra, $\partial_x f(x(t),p)$ dépend de
  l'état : la passe rétrograde doit interroger la trajectoire stockée `sol(t)` à chaque
  pas — ce qui ajoute un surcoût fixe (en temps et en mémoire), mais **indépendant de $m$**.
- Pourquoi les deux courbes gardent-elles une pente $1$ ici ? Parce que dans une EDO à
  paramètres constants $p$, les $m$ paramètres agissent à **chaque** pas de temps : il
  faut donc accumuler les $m$ composantes de $B(t)^\top\lambda(t)$ tout au long de la
  trajectoire. Dans un réseau de neurones à couches indépendantes (ou en contrôle
  optimal), chaque paramètre n'agit qu'à **sa** couche : le coût de la rétropropagation
  reste celui d'**une seule passe**, tandis que le mode direct doit propager une
  direction par paramètre — le rapport des coûts se compte alors en **millions**.
  L'annexe 8 détaille ce décompte.

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
8. Annexe — pourquoi « adjointe », et d'où vient le gain ?
</div>

*Cette annexe n'est pas nécessaire pour faire le TP : elle explique d'où vient le nom
« adjointe », pourquoi la construction de la section 5 n'a rien d'un tour de passe-passe,
quel est le lien avec la différentiation automatique par nombres duaux, et ce que change
le passage à des paramètres dépendant du temps (ResNet, contrôle optimal).*

## 8.1 Le cadre algébrique : un seul vecteur, trois parenthésages

Oublions le temps. Soit $f:\mathbb{R}^m\times\mathbb{R}^n\to\mathbb{R}$ un coût scalaire
et $g:\mathbb{R}^m\times\mathbb{R}^n\to\mathbb{R}^n$ une contrainte d'état. On veut
minimiser $f(\theta,x)$ sous $g(\theta,x)=0$. Notons les jacobiennes et le gradient
par rapport à l'état :

$$
M \;\coloneqq\; \partial_x g \ \in\ \mathbb{R}^{n\times n},
\qquad
N \;\coloneqq\; \partial_\theta g \ \in\ \mathbb{R}^{n\times m},
\qquad
v \;\coloneqq\; \nabla_x f \ \in\ \mathbb{R}^{n}.
$$

Si $M$ est inversible, le théorème des fonctions implicites définit localement $x(\theta)$
et la dérivation de $g(\theta,x(\theta))=0$ donne $N + M J = 0$, où
$J \coloneqq \partial_\theta x = -M^{-1}N \in \mathbb{R}^{n\times m}$. Le gradient du
coût réduit $F(\theta) = f(\theta,x(\theta))$ vaut alors, par la règle de la chaîne,

$$
\nabla F \;=\; \nabla_\theta f \;+\; J^\top v
\;=\; \nabla_\theta f \;-\; \underbrace{N^\top}_{m\times n}\;\underbrace{M^{-\top}}_{n\times n}\;\underbrace{v}_{n\times 1}.
$$

**Où l'adjoint apparaît.** Le terme $J^\top v$ provient de l'identité fondamentale du
produit scalaire : pour toute application linéaire $J$,

$$
\langle J u, v\rangle \;=\; \langle u, J^\top v\rangle ,
$$

et $J^\top$ s'appelle l'**adjoint** de $J$. À gauche, on transporte les directions
d'entrée $u \in \mathbb{R}^m$ **vers l'avant** (il y en a $m$ : *produits jacobien-vecteur*,
ou JVP). À droite, on transporte la seule direction de sortie $v \in \mathbb{R}^n$ **vers
l'arrière** (*produit vecteur-jacobien*, ou VJP). Le mode adjoint consiste à choisir le
côté le moins peuplé du produit scalaire.

**Trois parenthésages, trois coûts.** Le produit $-N^\top M^{-\top} v$ ne dépend pas de
l'ordre dans lequel on place les parenthèses (associativité), mais le coût en dépend
radicalement :

| méthode | ce qu'on calcule d'abord | objet formé | coût dominant |
|---|---|---|---|
| inverse explicite | $M^{-1}$, puis $J = -M^{-1}N$ | $n^2$ puis $nm$ nombres | $\Theta(n^3 + n^2m)$ |
| sensibilités (*forward*) | $J$ colonne par colonne : $M\,Je_j = -Ne_j$ | $nm$ nombres | $\Theta(n^3/3 + mn^2)$ |
| **adjoint** (*reverse*) | $\lambda$ tel que $M^\top\lambda = -v$, puis $N^\top\lambda$ | $n$ nombres | $\Theta(n^3/3 + n^2 + nm)$ |

Pour $m \gg n$, les sensibilités font $m$ résolutions (une par paramètre) alors que
l'adjoint n'en fait qu'**une seule**, et le rapport des coûts tend vers

$$
\frac{\text{sensibilités}}{\text{adjoint}} \;\approx\; \frac{m\,n^2}{n\,m} \;=\; \boxed{\,n\,}.
$$

**Pourquoi $\lambda$ est un multiplicateur de Lagrange.** Avec le lagrangien
$\mathcal{L}(\theta,x,\lambda) = f(\theta,x) + \langle\lambda,g(\theta,x)\rangle$, la
condition de stationnarité par rapport à l'état s'écrit
$\nabla_x\mathcal{L} = v + M^\top\lambda = 0$ : **résoudre l'équation adjointe et trouver
le multiplicateur de Lagrange sont le même calcul**, et $\nabla F = \nabla_\theta\mathcal{L} = \nabla_\theta f + N^\top\lambda$.

## 8.2 Le même calcul, avec des chiffres

Prenons tout linéaire, avec $n = 2$ états et $m = 3$ paramètres :

$$
g(\theta,x) = Mx + N\theta + c, \qquad f(\theta,x) = \langle v, x\rangle,
$$

$$
M = \begin{pmatrix} 2 & 1\\ 0 & 3\end{pmatrix},\qquad
N = \begin{pmatrix} 1 & 0 & 2\\ 0 & 1 & -1\end{pmatrix},\qquad
v = \begin{pmatrix} 2\\ 7\end{pmatrix},\qquad
\nabla_\theta f = 0 .
$$

**Route 1 — inverser.** $\det M = 6$, donc

$$
M^{-1} = \frac{1}{6}\begin{pmatrix} 3 & -1\\ 0 & 2\end{pmatrix},
\qquad
J = -M^{-1}N = \begin{pmatrix} -1/2 & 1/6 & -7/6\\ 0 & -1/3 & 1/3\end{pmatrix},
\qquad
\nabla F = J^\top v = \begin{pmatrix} -1\\ -2\\ 0\end{pmatrix}.
$$

**Route 2 — les sensibilités.** Sans former $M^{-1}$, on résout $M\,Je_j = -Ne_j$ pour
$j = 1,2,3$ — **trois** systèmes $2\times2$, un par paramètre :

$$
M\begin{pmatrix}s_1\\s_2\end{pmatrix} = \begin{pmatrix}-1\\0\end{pmatrix}
\ \Rightarrow\ \begin{pmatrix}-1/2\\0\end{pmatrix},
\qquad
\begin{pmatrix}0\\-1\end{pmatrix} \Rightarrow \begin{pmatrix}1/6\\-1/3\end{pmatrix},
\qquad
\begin{pmatrix}-2\\1\end{pmatrix} \Rightarrow \begin{pmatrix}-7/6\\1/3\end{pmatrix}.
$$

On retrouve les trois colonnes de $J$ ($nm = 6$ nombres), puis $\nabla F = J^\top v = (-1,-2,0)^\top$.

**Route 3 — l'adjoint.** On résout **un seul** système, transposé :

$$
M^\top\lambda = -v
\quad\Longleftrightarrow\quad
\begin{pmatrix} 2 & 0\\ 1 & 3\end{pmatrix}\begin{pmatrix}\lambda_1\\\lambda_2\end{pmatrix}
= \begin{pmatrix}-2\\-7\end{pmatrix}
\quad\Longrightarrow\quad
\lambda = \begin{pmatrix}-1\\-2\end{pmatrix},
$$

puis $\nabla F = N^\top\lambda = (-1,-2,0)^\top$, **sans que $J$ ait jamais existé**. Avec
$m = 10^4$ paramètres, la route 2 résout $10^4$ systèmes pour calculer $2\cdot10^4$
nombres, la route 3 résout **un** système de taille $2$.

## 8.3 Dans une EDO : la résolvante joue le rôle de $M^{-1}$

Revenons aux notations du TP : $x\in\mathbb{R}^n$, $p\in\mathbb{R}^m$,
$A(t) = \partial_xf(x(t),p)$, $B(t) = \partial_pf(x(t),p)$, résidu $v = x(T)-y$.

**La résolvante et la formule de Duhamel.** Notons $R(t,s) \in \mathbb{R}^{n\times n}$ la
résolvante de l'équation linéarisée $\dot z = A(t)z$, caractérisée par
$\partial_t R(t,s) = A(t)R(t,s)$ et $R(s,s) = I$. La solution de $\dot S = A(t)S + B(t)$,
$S(0) = 0$, s'écrit par la formule de Duhamel :

$$
S(T) \;=\; \int_0^T R(T,s)\,B(s)\,\mathrm{d}s .
$$

$R(T,s)B(s)\,\mathrm{d}s$ est l'effet mesuré en $T$ d'un coup de pouce $B(s)\delta p$
donné à l'instant $s$ et transporté jusqu'en $T$ par la dynamique linéarisée : **la
résolvante joue le rôle de $-M^{-1}$, et $B$ celui de $N$.**

**Le basculement.** En transposant sous l'intégrale, comme en 8.1 :

$$
\nabla J \;=\; S(T)^\top v
\;=\; \int_0^T B(s)^\top\, \underbrace{R(T,s)^\top v}_{\textstyle \eqqcolon\ \lambda(s)}\ \mathrm{d}s .
$$

Choisir l'adjoint, c'est calculer le vecteur $\lambda(s) = R(T,s)^\top v \in \mathbb{R}^n$
plutôt que la matrice $R(T,s)B(s) \in \mathbb{R}^{n\times m}$. En dérivant
$R(T,s)R(s,0) = R(T,0)$ par rapport à $s$, on obtient $\partial_s R(T,s) = -R(T,s)A(s)$,
d'où en transposant :

$$
\dot\lambda(s) = -A(s)^\top\lambda(s), \qquad \lambda(T) = R(T,T)^\top v = v .
$$

**Le sens rétrograde n'est pas un choix d'implémentation : $\lambda(s)$ est par
construction un transport depuis l'instant final $T$.**

| cas algébrique (section 8.1) | cas EDO (sections 3 et 5) |
|---|---|
| $M = \partial_xg$ | l'opérateur $\tfrac{\mathrm{d}}{\mathrm{d}t} - A(t)$ |
| $-M^{-1}$ | la résolvante $R(T,s)$ |
| $N = \partial_\theta g$ | $B(t) = \partial_pf$ |
| $J = -M^{-1}N$ ($m$ résolutions) | $S(T) = \int_0^T R(T,s)B(s)\,\mathrm{d}s$ (système augmenté) |
| $M^\top\lambda = -v$ (une résolution) | $-\dot\lambda = A^\top\lambda$, $\lambda(T) = v$ |
| $\nabla F = N^\top\lambda$ | $\nabla J = \int_0^T B^\top\lambda\,\mathrm{d}t$ |

## 8.4 Un exemple d'EDO entièrement explicite

Prenons $n = 2$, un forçage constant à $m = 2$ paramètres, et $A$ triangulaire :

$$
\dot x = Ax + p, \qquad
A = \begin{pmatrix} -1 & 0\\ 1 & -2\end{pmatrix}, \qquad
B = \partial_p f = I_2, \qquad T = \ln 2 .
$$

La résolvante vaut $R(u) = e^{Au} = \begin{pmatrix} e^{-u} & 0\\ e^{-u}-e^{-2u} & e^{-2u}\end{pmatrix}$,
avec $e^{-T} = \tfrac12$ et $e^{-2T} = \tfrac14$. Pour un résidu $v = x(T)-y = (8,\,8)^\top$ :

- **Par les sensibilités :** $S(T) = \int_0^T e^{Au}\,\mathrm{d}u = \begin{pmatrix} 1/2 & 0\\ 1/8 & 3/8\end{pmatrix}$,
  puis $\nabla J = S(T)^\top v = (5,\,3)^\top$.
- **Par l'adjoint :** sans former $S$, on remonte $\lambda(T-u) = e^{A^\top u}v = (16e^{-u}-8e^{-2u},\; 8e^{-2u})^\top$,
  et $\nabla J = \int_0^T \lambda\,\mathrm{d}u = (5,\,3)^\top$.

## 8.5 Le coût en détail sur le système de la section 7

Un solveur explicite à pas adaptatif (`Tsit5`, $6$ évaluations par pas) paie à chaque
pas le coût du second membre et de l'algèbre vectorielle proportionnelle à la dimension
de l'état intégré :

| poste | sensibilités | adjoint, passe directe | adjoint, passe rétrograde |
|---|---|---|---|
| état intégré | $(x,S)$ : $\;n+nm$ | $x$ : $\;n$ | $(\lambda,g)$ : $\;n+m$ |
| forçage $\sum_j p_j\varphi_j(t)$ | $\Theta(m)$ | $\Theta(m)$ | — |
| $A\,S$ (matrice $\times$ matrice) | $n^2m$ | — | — |
| $A^\top\lambda$ (matrice $\times$ vecteur) | — | — | $n^2$ |
| $B(t)^\top\lambda$ | — | — | $nm$ |
| **par évaluation** | $\Theta(n^2 m)$ | $\Theta(n^2 + m)$ | $\Theta(n^2 + nm)$ |

Les nombres de pas (`length(sol.t)`) ne dépendent pas de la machine et permettent de
vérifier directement le modèle « $N \times \dim(\text{état})$ » avec $n = 20$ :

| $m$ | $\dim$ état sens. | pas $N_S$ | $\dim$ état adj. | pas $N_\lambda$ | $N_S(n{+}nm)$ | $N_f\,n + N_\lambda(n{+}m)$ | rapport prédit |
|---|---|---|---|---|---|---|---|
| $2$ | $60$ | $60$ | $22$ | $47$ | $3\,600$ | $2\,034$ | $1{,}8$ |
| $32$ | $660$ | $54$ | $52$ | $46$ | $35\,640$ | $3\,412$ | $10{,}4$ |
| $128$ | $2\,580$ | $53$ | $148$ | $44$ | $136\,740$ | $7\,532$ | $18{,}2$ |
| $512$ | $10\,260$ | $47$ | $532$ | $40$ | $482\,220$ | $22\,300$ | $21{,}6$ |
| $1024$ | $20\,500$ | $43$ | $1\,044$ | $38$ | $881\,500$ | $40\,692$ | $21{,}7$ |

Grâce au forçage lisse $\sin(jt)/j^3$, le nombre de pas reste voisin de $40$–$60$ quel
que soit $m$, et le rapport de travail converge vers **$n = 20$**.

## 8.6 Sensibilités, nombres duaux et méthode adjointe : les trois niveaux

En incluant la différentiation automatique en mode direct (nombres duaux), il existe
**trois** régimes de coût pour calculer $\nabla J$ (avec $C_f$ le coût d'une évaluation
de $f$) :

| approche | ce qui est fait | coût sur $N$ pas |
|---|---|---|
| **1. Mode direct naïf** (ou 1 nombre dual scalaire par paramètre) | $m$ intégrations séparées de $(x, s_j)$ : la trajectoire $x(t)$ est recalculée $m$ fois | $N\,m\,(C_f + n^2)$ |
| **2. Système augmenté** $(x,S)$ (ou nombre dual vectoriel *chunk*) | 1 seule intégration de taille $n+nm$ : $x(t)$ et $A(x)$ sont **mutualisés** entre les $m$ colonnes | $N\,(C_f + n^2 m)$ |
| **3. Méthode adjointe** (mode *reverse*) | 1 passe directe de taille $n$, 1 passe rétrograde de taille $n+m$ | $N\,(C_f + n^2 + nm)$ |

**Quelle différence entre le système augmenté $\dot S = AS+B$ et les nombres duaux (`ForwardDiff.jl`) ?**
Tous deux relèvent du mode direct (*forward*), mais :

- **Système augmenté = *dériver puis discrétiser* (*Differentiate-then-Discretize*) :** on
  écrit l'EDO variationnelle continue à la main, puis on la confie au solveur numérique,
  qui contrôle l'erreur sur le couple $(x,S)$ et profite du produit matriciel BLAS $A\,S$.
- **Nombres duaux = *discrétiser puis dériver* (*Discretize-then-Differentiate*) :** on
  passe directement des nombres duaux $p_j + \varepsilon_j$ au solveur `solve(prob, Tsit5())`.
  L'arithmétique duale $f(x+\varepsilon s, p+\varepsilon e_j) = f(x,p) + \varepsilon(\partial_x f\,s + \partial_p f\,e_j)$
  propage la même équation variationnelle à l'intérieur des étages de Runge-Kutta (et
  dérive même le choix des pas adaptatifs $\Delta t$). Passés un par un, les nombres duaux
  tombent dans le cas 1 (coût $\times m$) ; regroupés en vecteur (*chunk mode*), ils
  reviennent au cas 2.

## 8.7 Paramètres dépendant du temps $p(t)$ : ResNet et contrôle optimal

Comme vu en section 6, un ResNet générique (un jeu de poids $\beta_k$ **par couche**) a
pour analogue continu un problème de **contrôle optimal** où le paramètre est une
fonction du temps $p(\cdot) : [0,T]\to\mathbb{R}^d$ :

$$
\dot x(t) = f\big(x(t),p(t)\big).
$$

La même intégration par parties qu'en section 5 donne alors

$$
\delta J = \int_0^T \lambda(t)^\top B(t)\,\delta p(t)\,\mathrm{d}t,
\qquad -\dot\lambda = A^\top\lambda,\quad \lambda(T) = x(T)-y .
$$

Lorsque $\delta p(t)$ est libre de varier indépendamment à chaque instant $t$, on ne
factorise plus $\delta p$ hors de l'intégrale : le gradient fonctionnel est la **valeur
locale** $\nabla_{p(t)} J = B(t)^\top\lambda(t)$, et après discrétisation sur $N$ étapes
($M = Nd$ paramètres au total) :

$$
\nabla_{p_k} J \;=\; h\,B(t_k)^\top\lambda(t_k) \qquad (\text{aucun accumulateur } g \text{ à intégrer}).
$$

Comparons alors les coûts sur $N$ couches de largeur $n$, avec $d$ paramètres par couche :

- **Adjoint (rétropropagation) :** une passe directe, puis une passe rétrograde qui à
  chaque étape $k$ calcule $A_k^\top\lambda$ ($O(n^2)$) et lit $B_k^\top\lambda$ ($O(nd)$),
  soit $\Theta\big(N(n^2 + nd)\big)$ au total.
- **Sensibilités (*forward*) :** les $d$ paramètres de la couche $k$ créent une matrice
  de sensibilité $S_k \in \mathbb{R}^{n\times d}$ qu'il faut propager sur les $N-k$
  couches restantes au prix de $\Theta\big(n^2 d (N-k)\big)$. En sommant sur
  $k = 1,\dots,N$, on paie $\Theta(n^2 d N^2) = \Theta(n^2 M N)$ (et $\Theta(N M C_f)$ en
  mode direct naïf direction par direction).

| aspect | $p$ constant (estimation de paramètres / RNN) | $p_k$ par étape (contrôle optimal / ResNet) |
|---|---|---|
| paramètres | $m$ partagés à tous les pas | $M = Nd$ ($d$ par étape, indépendants) |
| gradient | accumulé : $\int_0^T B^\top\lambda\,\mathrm{d}t$ | local : $B_k^\top\lambda_{k+1}$ |
| gain sensibilités matricielles / adjoint | $\Theta(n)$ | $\Theta\big(N \min(n,d)\big)$ |
| gain forward naïf / adjoint | $\Theta(m)$ | $\Theta(M) = \Theta(Nd)$ |

In [ ]:
# --- annexe 8.5 : vérification par les nombres de pas et la dimension de l'état (n = 20) ---
@printf("%5s | %8s %6s | %8s %6s | %11s %11s | %8s\n",
        "m", "dim sens", "pas", "dim adj", "pas", "N·dim sens", "N·dim adj", "rapport")
for m ∈ [2, 32, 128, 512, 1024]
    p_ann = 0.1 * ones(m)

    sol_S = solve(ODEProblem(osc_augmente, vcat(x0_osc, zeros(n_osc * m)), (0.0, T_osc), p_ann,
                             reltol = 1e-8, abstol = 1e-8), Tsit5())
    sol_f = solve(ODEProblem(oscillateur, x0_osc, (0.0, T_osc), p_ann,
                             reltol = 1e-8, abstol = 1e-8), Tsit5())
    sol_λ = solve(ODEProblem(osc_adjoint, vcat(sol_f(T_osc) - y_osc, zeros(m)), (T_osc, 0.0),
                             (p_ann, sol_f), reltol = 1e-8, abstol = 1e-8), Tsit5())

    trav_S = length(sol_S.t) * (n_osc + n_osc * m)
    trav_A = length(sol_f.t) * n_osc + length(sol_λ.t) * (n_osc + m)

    @printf("%5d | %8d %6d | %8d %6d | %11d %11d | %8.2f\n",
            m, n_osc + n_osc * m, length(sol_S.t), n_osc + m, length(sol_λ.t),
            trav_S, trav_A, trav_S / trav_A)
end